# Chat with Documents (Open-Source LLMs)

A question-answering app over your own **PDF documents**, powered by **open-source LLMs** loaded and run locally via HuggingFace — no proprietary API required.

**What it demonstrates**
- Loading open-source models with Transformers and running inference
- Extracting and chunking PDF text
- Building a document-chat UI with Gradio

**Stack:** Python · HuggingFace Transformers · PyTorch · pypdf · Gradio


In [1]:
import torch
import pypdf
import gradio as gr
from IPython.display import display, Markdown
import os
from dotenv import load_dotenv
print("Core libraries imported.")

Core libraries imported.


In [2]:
from huggingface_hub import login

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

login(token=hf_token)

print("Logged into Hugging Face Hub successfully!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged into Hugging Face Hub successfully!


In [3]:
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    # Set default device to GPU
    torch.set_default_device("cuda")
    print("PyTorch default device set to CUDA (GPU).")
else:
    print("WARNING: No GPU detected. Running these models on CPU will be extremely slow!")
    print("Make sure 'GPU' is selected in Runtime > Change runtime type.")

GPU detected: NVIDIA GeForce RTX 4080 Laptop GPU
PyTorch default device set to CUDA (GPU).


In [4]:
# Helper function for markdown display
def print_markdown(text):
    """Displays text as Markdown in Colab/Jupyter."""
    display(Markdown(text))

In [5]:
from transformers import pipeline

pipe = pipeline(model = "ProsusAI/finbert")
pipe("Apple lost 10 Million dollars today due to US tarrifs")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'negative', 'score': 0.9706032276153564}]

In [6]:
pipe("Microsoft is scheduled to release its earnings report next week.")

[{'label': 'neutral', 'score': 0.8829190135002136}]

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokens = tokenizer("Hello, everyone! I am an AI Engineer. I hope to make a lot of money in the future")

print(tokens["input_ids"])

[15496, 11, 2506, 0, 314, 716, 281, 9552, 23164, 13, 314, 2911, 284, 787, 257, 1256, 286, 1637, 287, 262, 2003]


In [8]:
token_text = tokenizer.tokenize("Hello, everyone! I am an AI Engineer. I hope to make a lot of money in the future")
print(token_text)

['Hello', ',', 'Ġeveryone', '!', 'ĠI', 'Ġam', 'Ġan', 'ĠAI', 'ĠEngineer', '.', 'ĠI', 'Ġhope', 'Ġto', 'Ġmake', 'Ġa', 'Ġlot', 'Ġof', 'Ġmoney', 'Ġin', 'Ġthe', 'Ġfuture']


In [9]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model_id = "microsoft/Phi-4-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
print("Tokenizer loaded successfully.")



[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Tokenizer loaded successfully.


In [10]:
print(f'Loading model {model_id}')
print("This may take a few minutes especially for the first time...")

quantization_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_use_double_quant=True,
                                         bnb_4bit_quant_type="nf4")

model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config=quantization_config,
                                             device_map="auto")



Loading model microsoft/Phi-4-mini-instruct
This may take a few minutes especially for the first time...


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

In [11]:
prompt = "Explain how Electric Vehicles work in a funny way!"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=1000)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print_markdown(response)

Explain how Electric Vehicles work in a funny way! Electric vehicles, or EVs, are like the superheroes of the car world, but instead of capes, they come equipped with batteries. Imagine your car is a little robot, and instead of gasoline, it drinks electricity like a vampire drinks blood. 

First, the battery pack, which is like a giant energy sandwich, gets charged up at home using a regular power outlet. Think of it as plugging in your phone, but instead of a few hours, it can take a whole night to fully charge up. 

Once the battery is full, the car's brain, or the onboard computer, wakes up and starts the engine. But instead of roaring like a gas-powered car, it hums quietly, like a ninja in stealth mode. The electric motor, which is the heart of the EV, then kicks in, spinning the wheels with the power of electricity. 

The car glides along, powered by electricity, and you can feel the wind in your hair as it zips by. It's like riding a silent, eco-friendly roller coaster. 

And the best part? EVs don't need to stop for gas, so you can just keep cruising and save the planet. It's like having a magic carpet ride, but without the annoying noise and pollution. 

So, next time you see an EV zooming by, just remember, it's like a superhero car powered by electricity, ready to save the day and the environment!

In [12]:
pipe = pipeline("text-generation", 
                model = model,
                tokenizer = tokenizer,
                torch_dtype = "auto",
                device_map="auto")
output = pipe(prompt,
              max_new_tokens=1000,
              temperature = 1)

print_markdown(output[0]["generated_text"])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=Tr

Explain how Electric Vehicles work in a funny way! Electric Cars or Vehicles (EVs) are kind of like Transformers but without the cool fight scenes. Imagine an electric car as a secret superhero, but instead of blasting buildings, it’s blasting away pollutants and using batteries to power its superpowers.

Inside this EV, there's a battery pack made of thousands of little energy-carrying units. It's like having a tiny army of Pac-Man-style dots rolling around inside the car, gobbling up energy as they travel. These dots are hooked onto a battery bus — imagine it like a bus full of energy carriers, all getting around the city.

When you're driving an electric car, it's like you're in a race where the fuel comes from a battery pack rather than gas. You plug into a power station, and Voila! The car is fueled up and ready for action. The car's brain is a smart controller, coordinating everything like a director yelling, “Action!”

Now, imagine the wheels being like silent ninjas, spinning and moving without making a sound. They're super efficient, almost mystical in their operation, and work together to propel the car forward. These ninjas (wheels) are powered by what we call 'electrification' — a fancy term for replacing energy from gasoline with electricity.

When you stop, there's an electric motor that spins up to drive the car. It's like having a mini motor running in your pocket, ready to power up whenever you need it. And then, there's a plug that sits in the charging port, waiting like a giant battery that fills up whenever it plugs in. Easy peasy, right?

Electric cars need charging stations, which are like gas stations, but instead of filling up with gasoline, these chargers pour energy back into the car’s batteries, refueling it for another epic journey.

So next time you hear about electric vehicles, chuckle at its simplicity, and remember it's like having a little secret superhero team working together to make sure you get from A to B, leaving less pollution in its trail!

In [13]:
import requests
from pathlib import Path

pdf_url = "https://abc.xyz/assets/66/ae/c94682fc4137b5fb90a5d709ac4b/2025-q1-earnings-transcript.pdf"
pdf_filename = "google_earning_transcript.pdf"
pdf_path = Path(pdf_filename)

if not pdf_path.exists():
    response = requests.get(pdf_url)
    response.raise_for_status()
    pdf_path.write_bytes(response.content)
    print(f"Downloaded PDF to {pdf_path}")

else:
    print(f"PDF already exists at {pdf_path}")


pdf_text = ""

reader = pypdf.PdfReader(str(pdf_path))
num_pages = len(reader.pages)
print(f"Number of pages in PDF: {num_pages}")

all_pages_text = []

for i, pages in enumerate(reader.pages):
    page_text = pages.extract_text()
    if page_text:
        all_pages_text.append(page_text)

pdf_text = "\n".join(all_pages_text)

print(f"Extracted text from PDF. Total characters: {len(pdf_text)}")

Ignoring wrong pointing object 572 0 (offset 0)


PDF already exists at google_earning_transcript.pdf
Number of pages in PDF: 21
Extracted text from PDF. Total characters: 64652


In [14]:
print("\n--- Snippet of Extracted Text ---")
print_markdown(f"{pdf_text[-1000:]}")


--- Snippet of Extracted Text ---


  
celebrated
 
its
 
20th
 
birthday
 
and
 
we
 
now
 
have
 
more
 
than
 
20 billion
 
videos
 
on
 
YouTube,
 
and
 
we
 
get
 
20 million
 
videos
 
uploaded
 
every
 
day.
  So,  I  think  it's  a  tremendous  platform,  and  thanks  to  all  the  creators  and  users  who  have  
supported
 
us
 
there
 
over
 
the
 
years.
 
  Ron  Josey  (Citi):  Great,  thank  you.    Operator:  Thank  you.  And  that  concludes  our  question-and-answer  session  for  today.    I  would  like  to  turn  the  conference  back  over  to  Jim  Friedland  for  any  further  remarks.    Jim  Friedland,  Senior  Director,  Investor  Relations:  Thanks,  everyone,  for  joining  us  today.  
We
 
look
 
forward
 
to
 
speaking
 
with
 
you
 
again
 
on
 
our
 
second
 
quarter
 
2025
 
call.
 
Thank
 
you,
 
and
 
have
 
a
 
good
 
evening.
 
  Operator:  Thank  you,  everyone.  This  concludes  today's  conference  call.  Thank  you  for  
participating.
 
You
 
may
 
now
 
disconnect.
 
   
21  

In [18]:
MAX_CONTEXT_CHARS = 6000


def answer_question_from_pdf(document_text, question, llm_pipeline):

    """
    Answers a question based on the provided document text using the loaded LLM pipeline.

    Args:
        document_text (str): The text extracted from the PDF.
        question (str): The user's question.
        llm_pipeline (transformers.pipeline): The initialized text-generation pipeline.

    Returns:
        str: The model's generated answer.
    """
    # Truncate context if necessary
    if len(document_text) > MAX_CONTEXT_CHARS:
        print(f"Warning: Document text ({len(document_text)} chars) exceeds limit ({MAX_CONTEXT_CHARS} chars). Truncating.")
        context = document_text[:MAX_CONTEXT_CHARS] + "..."
    else:
        context = document_text

    prompt_template = f"""<|system|>
    You are an AI assistant. Answer the following question based *only* on the provided document text. If the answer is not found in the document, say "The document does not contain information on this topic." Do not use any prior knowledge.

    Document Text:
    ---
    {context}
    ---
    <|end|>
    <|user|>
    Question: {question}<|end|>
    <|assistant|>
    Answer:""" 

    print(f"\n...Generating answer for: {question} ...\n")

    output = llm_pipeline(prompt_template, 
                          max_new_tokens=500,
                          do_sample=True,
                          temperature=0.2,
                          top_p=0.9)
    
    full_generated_text = output[0]["generated_text"]
    answer_start_index = full_generated_text.find("Answer:") + len("Answer:")
    raw_answer = full_generated_text[answer_start_index:].strip()

    end_token = "<|end|>"
    if end_token in raw_answer:
            raw_answer = raw_answer.split(end_token)[0]

    print("--- Generation Complete ---")
    return raw_answer

In [16]:
import torch
import gc

def answer_question_from_pdf(document_text, question, llm_pipeline):
    # Clear GPU memory first
    torch.cuda.empty_cache()
    gc.collect()
    
    # rest of your code...

In [19]:
test_question = "What is this document about?"

generated_answer = answer_question_from_pdf(pdf_text, test_question, pipe)

print("\n Test Question: ", test_question)
print("\n Generated Answer:")
print_markdown(generated_answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



...Generating answer for: What is this document about? ...

--- Generation Complete ---

 Test Question:  What is this document about?

 Generated Answer:


This document is about Alphabet's First Quarter 2025 Earnings Conference Call, where Jim Friedland, Senior Director of Investor Relations, presents the company's financial performance and forward-looking statements. CEO Sundar Pichai also discusses the company's AI progress, infrastructure, and product innovations. The document includes a reconciliation of non-GAAP to GAAP financial measures and mentions the company's strong results in various business segments, including AI-powered features, Subscriptions, and Cloud growth. The document is intended for investors and includes a link to the Investor Relations website for further information.

In [28]:
available_models = {
    "Llama 3.2": "unsloth/Llama-3.2-3B-Instruct",
    "Microsoft Phi-4 Mini": "microsoft/Phi-4-mini-instruct",
    "Google Gemma 3": "unsloth/gemma-3-4b-it",
    "Qwen 2.5": "Qwen/Qwen2.5-3B-Instruct"
    }

In [29]:
# --- Global State (or use gr.State in Blocks) ---
# To keep track of the currently loaded model/pipeline
current_model_id = None
current_pipeline = None
print(f"Models available for selection: {list(available_models.keys())}")


# Define a function to Load/Switch Models
def load_llm_model(model_name):
    """Loads the selected LLM, unloading the previous one."""
    global current_model_id, current_pipeline, tokenizer, model

    new_model_id = available_models.get(model_name)
    if not new_model_id:
        return "Invalid model selected.", None  # Return error message and None pipeline

    if new_model_id == current_model_id and current_pipeline is not None:
        print(f"Model {model_name} is already loaded.")
        # Indicate success but don't reload
        return f"{model_name} already loaded.", current_pipeline

    print(f"Switching to model: {model_name} ({new_model_id})...")

    # Unload previous model (important for memory)
    # Clear variables and run garbage collection
    current_pipeline = None
    if "model" in locals():
        del model
    if "tokenizer" in locals():
        del tokenizer
    if "pipe" in locals():
        del pipe
    torch.cuda.empty_cache()  # Clear GPU memory cache
    import gc

    gc.collect()
    print("Previous model unloaded (if any).")

    # --- Load the new model ---
    loading_message = f"Loading {model_name}..."
    try:
        # Load Tokenizer
        tokenizer = AutoTokenizer.from_pretrained(new_model_id, trust_remote_code = True)

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        model = AutoModelForCausalLM.from_pretrained(
            new_model_id,
            quantization_config=quantization_config,  # ✅ Correct way
            device_map="auto",
            trust_remote_code=True
        )

        # Load Model (Quantized)
    #model = AutoModelForCausalLM.from_pretrained(new_model_id,
     #                                               torch_dtype = "auto",  # "torch.float16", # Or bfloat16 if available
      #                                              load_in_4bit = True,
       #                                             device_map = "auto",
        #                                            trust_remote_code = True)

        # Create Pipeline
        loaded_pipeline = pipeline(
            "text-generation", model = model, tokenizer = tokenizer, torch_dtype = "auto", device_map = "auto")

        print(f"Model {model_name} loaded successfully!")
        current_model_id = new_model_id
        current_pipeline = loaded_pipeline  # Update global state
        # Use locals() or return values with gr.State for better Gradio practice
        return f"{model_name} loaded successfully!", loaded_pipeline  # Status message and the pipeline object

    except Exception as e:
        print(f"Error loading model {model_name}: {e}")
        current_model_id = None
        current_pipeline = None
        return f"Error loading {model_name}: {e}", None  # Error message and None pipeline

Models available for selection: ['Llama 3.2', 'Microsoft Phi-4 Mini', 'Google Gemma 3', 'Qwen 2.5']


In [30]:
# --- Function to handle Q&A Submission ---
def handle_submit(question):
    """Handles the user submitting a question."""
    if not current_pipeline:
        return "Error: No model is currently loaded. Please select a model."
    if not pdf_text:
        return "Error: PDF text is not loaded. Please run Section 4."
    if not question:
        return "Please enter a question."

    print(f"Handling submission for question: '{question}' using {current_model_id}")
    # Call the Q&A function defined in Section 5
    answer = answer_question_from_pdf(pdf_text, question, current_pipeline)
    return answer

In [31]:

# --- Build Gradio Interface using Blocks ---
print("Building Gradio interface...")
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        f"""
    # PDF Q&A Bot Using Hugging Face Open-Source Models
    Ask questions about the document ('{pdf_filename}' if loaded, {len(pdf_text)} chars).
    Select an open-source LLM to answer your question.
    **Note:** Switching models takes time as the new model needs to be downloaded and loaded into the GPU.
    """
    )

    # Store the pipeline in Gradio state for better practice (optional for this simple version)
    # llm_pipeline_state = gr.State(None)

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=list(available_models.keys()),
            label="🤖 Select LLM Model",
            value=list(available_models.keys())[0],  # Default to the first model
        )
        status_textbox = gr.Textbox(label="Model Status", interactive=False)

    question_textbox = gr.Textbox(
        label="❓ Your Question", lines=2, placeholder="Enter your question about the document here..."
    )
    submit_button = gr.Button("Submit Question", variant="primary")
    answer_textbox = gr.Textbox(label="💡 Answer", lines=5, interactive=False)

    # --- Event Handlers ---
    # When the dropdown changes, load the selected model
    model_dropdown.change(
        fn = load_llm_model,
        inputs = [model_dropdown],
        outputs = [status_textbox],  # Update status text. Ideally also update a gr.State for the pipeline
        # outputs=[status_textbox, llm_pipeline_state] # If using gr.State
    )

    # When the button is clicked, call the submit handler
    submit_button.click(
        fn = handle_submit,
        inputs = [question_textbox],
        outputs = [answer_textbox],
        # inputs=[question_textbox, llm_pipeline_state], # Pass state if using it
    )

    # --- Initial Model Load ---
    # Easier: Manually load first model *before* launching Gradio for simplicity here
    initial_model_name = list(available_models.keys())[0]
    print(f"Performing initial load of default model: {initial_model_name}...")
    status, _ = load_llm_model(initial_model_name)
    status_textbox.value = status  # Set initial status
    print("Initial load complete.")


# --- Launch the Gradio App ---
print("Launching Gradio demo...")
demo.launch(debug=True)  # debug=True provides more detailed logs

Building Gradio interface...
Performing initial load of default model: Llama 3.2...
Switching to model: Llama 3.2 (unsloth/Llama-3.2-3B-Instruct)...


C:\Users\princ\AppData\Local\Temp\ipykernel_16412\885397300.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Previous model unloaded (if any).


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model Llama 3.2 loaded successfully!
Initial load complete.
Launching Gradio demo...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


...

Model Google Gemma 3 loaded successfully!


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Handling submission for question: 'What is the document about?
' using unsloth/gemma-3-4b-it

...Generating answer for: What is the document about?
 ...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- Generation Complete ---
Keyboard interruption in main thread... closing server.
